# HW 4: Dynamic Programming

Code up the dynamic programming problem for consumers who are forward-looking about getting "locked-in" to one of the yogurt brands.

To keep things simple, ignore feature and price, assuming the only explanatory variable is the brand they are locked-in to.

Keep in mind that the actual data set should play no role in your coding, other than understanding the nature of the problem.  You are simply calculating the future/continuation values associated with each choice.

In [1]:
import pandas as pd
import numpy as np
from numpy import euler_gamma
from scipy.special import logsumexp

In [2]:
# Parameters (from homogeneous MNL; see notebook 2)

beta_0 = np.array([0, -0.699019, -1.392080, -3.825416, -1.254406])
beta_b = np.array([2.466313])

delta = 0.99

In [3]:
states = np.array([0, 1, 2, 3, 4])    # what brand is the customer "locked-in" to?
actions = np.array([0, 1, 2, 3, 4])   # what brand does the customer choose to buy?

In [4]:
def value_function(Nu):
    """
    Calculates the value function given the current guess of Nu.

    u(a,s) = beta_0^T 1{a} + beta_b 1{a=s}
    U_ij = u(i,j)

    nu(a,s) = u(a,s) + delta * (lambda + log-sum-exp_{a'}(v(a',s)))
    Nu_ij = nu(i,j)
    
    Shape: (n_a, n_s) = (5, 5)
    """
    U = beta_0[:, None] + beta_b * np.eye(5)
    
    # the action taken becomes the state next period
    Nu_fut = logsumexp(Nu, axis=0).T
    Nu_curr = U + delta * (euler_gamma + Nu_fut[:, None])
    return Nu_curr

def find_fixed_point(Nu_init, tol=1e-6):
    Nu_curr = Nu_init
    diff = np.inf
    while diff > tol:
        Nu_next = value_function(Nu_curr)
        diff = np.linalg.norm(Nu_next - Nu_curr, ord='fro')
        Nu_curr = Nu_next
    return Nu_curr

In [5]:
Nu_init = np.full((5, 5), 0)
Nu = find_fixed_point(Nu_init)
Nu = pd.DataFrame(Nu)
print(Nu)

            0           1           2           3           4
0  305.139287  302.672974  302.672974  302.672974  302.672974
1  300.271976  302.738289  300.271976  300.271976  300.271976
2  299.232924  299.232924  301.699237  299.232924  299.232924
3  296.562225  296.562225  296.562225  299.028538  296.562225
4  299.414830  299.414830  299.414830  299.414830  301.881143
